In [ ]:
import pandas as pd
from rail_connectors.connection import connect

excel_path = "/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx"
month_start = "2026-03-01"
month_end = "2026-03-31"

def norm_agr(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def norm_inn(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
        .str.replace(r"\\D", "", regex=True)
        .where(lambda x: x.str.len().isin([10, 12]), pd.NA)
    )

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()


In [ ]:
# Секция 1: 10 ИНН из Excel (есть agr_id), но нет agr_id в ods_alpha.scd1_agreements за март
excel = pd.read_excel(excel_path).copy()
excel['agr_id_norm'] = norm_agr(excel['ID договора'])
excel['inn_norm'] = norm_inn(excel['ИНН'])

with imp:
    agr = imp.fetch(f"""
        select distinct cast(abs_agr_id as string) as agr_id
        from ods_alpha.scd1_agreements
        where abs_agr_id is not null
          and cast(d_valid_from as date) <= cast('{month_end}' as date)
          and (d_valid_to is null or cast(d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(ods_deleted_flg, '0') <> '1'
    """)

agr_set = set(norm_agr(agr['agr_id']).dropna().unique().tolist()) if agr is not None else set()

inn_list_1 = (
    excel[
        excel['agr_id_norm'].notna() &
        (~excel['agr_id_norm'].isin(agr_set)) &
        excel['inn_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .sort_values()
    .head(10)
    .tolist()
)

for i, inn in enumerate(inn_list_1, 1):
    print(f"{i}. {inn}")


In [ ]:
# Секция 2: 10 ИНН по agr_id из r2_ip_merchants (март по датам), которых нет в agreements за март
with imp:
    r2 = imp.fetch(f"""
        select distinct
            cast(m.id as string) as agr_id,
            regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_client cl
          on cast(cl.id as string) = cast(m.c_cl_org as string)
        where cast(m.c_date_begin as date) <= cast('{month_end}' as date)
          and (
                m.c_date_close is null
             or cast(m.c_date_close as date) >= cast('{month_start}' as date)
          )
    """)

if r2 is None:
    r2 = pd.DataFrame(columns=['agr_id', 'inn'])

r2['agr_id_norm'] = norm_agr(r2['agr_id'])
r2['inn_norm'] = norm_inn(r2['inn'])

inn_list_2 = (
    r2[
        r2['agr_id_norm'].notna() &
        (~r2['agr_id_norm'].isin(agr_set)) &
        r2['inn_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .sort_values()
    .head(10)
    .tolist()
)

for i, inn in enumerate(inn_list_2, 1):
    print(f"{i}. {inn}")
